**Objective:** model and interpret the drivers of country-level mobile money adoption rates (`mobileaccount_t_d`).

In [44]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Modeling & stats
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Diagnostics
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Quick plotting style
sns.set_theme(style="whitegrid")


In [ ]:
# Load data
df = pd.read_csv('../data/processed/global_findex_cleaned.csv')
df.head()

In [ ]:
# Quick check (shape / columns / dtypes / missing)
print("shape:", df.shape)
print("\nColumns (sample):", df.columns.tolist()[:40])
print("\nMissing per columns (top 20):")
print(df.isnull().sum().sort_values(ascending=False).head(20))
df.describe(include='all').T

# Choose target & candidate predictors.
**Target (y):** `mobileaccount_t_d` (proportion 0-1)
**Candidate predictors (x):**
- `account_t_d`
- `year`
- `regionwb24_hi` (region)
- `incomegroupwb24` (income group)
- `pop_adult`
- `internet`


In [25]:
# Select relevant columns
cols = ['year', 'regionwb24_hi', 'incomegroupwb24', 'pop_adult',
        'account_t_d', 'mobileaccount_t_d', 'internet']
df_model = df[cols].copy()
print("shape:", df_model.shape)
df_model.head()

shape: (7880, 7)


,year,regionwb24_hi,incomegroupwb24,pop_adult,account_t_d,mobileaccount_t_d,internet
0,2011,South Asia (excluding high income),Low income,14575546.0,0.090050,NaN,NaN
1,2011,Europe & Central Asia (excluding high income),Upper middle income,2281010.0,0.282681,NaN,NaN
2,2011,Middle East & North Africa (excluding high inc...,Lower middle income,26251587.0,0.332861,NaN,NaN
3,2011,Sub-Saharan Africa (excluding high income),Lower middle income,12779501.0,0.392035,NaN,NaN
4,2011,Latin America & Caribbean (excluding high income),Upper middle income,30685516.0,0.331302,NaN,NaN


In [ ]:
# Handle missing data

# Show how many rows have missing target
print("Rows with missing target:", df_model['mobileaccount_t_d'].isnull().sum())

# Inspect missing pattern for predictors
print("\nMissing per columns:")
print(df_model.isnull().sum().sort_values(ascending=False))

In [ ]:
# Impute missing values in the target variable
df_model['mobileaccount_t_d'] = df_model['mobileaccount_t_d'].bfill(axis=0).fillna(0)

# Impute missing values in predictors
df_model['internet'] = df_model['internet'].bfill(axis=0).fillna(0)
df_model['account_t_d'] = df_model['account_t_d'].fillna(df_model['account_t_d'].median())

# Confirm no missing values remain
print("\nMissing per columns after imputation:")
print(df_model.isnull().sum().sort_values(ascending=False))

In [ ]:

# Encode categorical variables

# Check data types
print(df_model.dtypes)

# One-hot encode categorical variables
categorical_cols = ['regionwb24_hi', 'incomegroupwb24']
df_model_encoded = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

print("\nShape before encoding:", df_model.shape)
print("\nShape after encoding:", df_model_encoded.shape)
df_model_encoded.head()

In [ ]:
# Split features and target, train-test split
X = df_model_encoded.drop(columns=['mobileaccount_t_d'])
y = df_model_encoded['mobileaccount_t_d']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("--- Data Split Confirmation ---")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
print("--- Normalization (Standardization) ---")
# Exclude categorical variables from scaling
numerical_cols = ['year', 'pop_adult', 'account_t_d', 'internet']

# Initialize standard scaler
standard_scaler = StandardScaler()

# Fit and transform training data, transform test data
X_train[numerical_cols] = standard_scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = standard_scaler.transform(X_test[numerical_cols])

print("\nVerification of standardization (mean ~0, std ~1):")

# Check the Training Data (X_train)

# Calculate and print the mean for each standardized column 
print("\nMean of standardized numerical columns (should be near zero):")
print(X_train[numerical_cols].mean().round(4))

# Calculate and print the standard deviation for each standardized column 
print("\nStandard deviation of standardized numerical columns (should be near one):")
print(X_train[numerical_cols].std().round(4))

# Check the Testing Data (X_test)

# Calculate and print the mean for each standardized column in test set
print("\nMean of standardized numerical columns in test set (should be near zero):")
print(X_test[numerical_cols].mean().round(4))

# Calculate and print the standard deviation for each standardized column in test set
print("\nStandard deviation of standardized numerical columns in test set (should be near one):")
print(X_test[numerical_cols].std().round(4))





In [57]:
# Train the model, make predictions, and evaluate its initial performance

# 1. Initialize the model
model = LinearRegression()

# 2. Train the model
model.fit(X_train, y_train)
print("Model trained.")

# 3. Make predictions
y_pred = model.predict(X_test)

# 4. Evaluate the model performance
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

# Print evaluation metrics
print("\n--- Model Evaluation ---")
print(f"R-squared: {r2:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")


# Save model evaluation summary to a text file
# Create the content for the text file
summary_text = (
    "--- Linear Regression V1: Correlated Variables ---\n"
    f"R-squared: {r2:.4f}\n"
    f"Mean Squared Error (MSE): {mse:.4f}\n"
    f"Root Mean Squared Error (RMSE): {rmse:.4f}\n"
    "\nNOTE: This model contains correlated independent variables (multicollinearity)."
)

# Define the path to save the summary
output_dir = "../outputs/model_summaries/"
os.makedirs(output_dir, exist_ok=True)
file_name = 'linear_model_summary_V1_correlated.txt'
full_path = os.path.join(output_dir, file_name)
with open(full_path, 'w') as f:
    f.write(summary_text)

print(f"Summary saved: {full_path}")



Model trained.

--- Model Evaluation ---
R-squared: 0.1184
Mean Squared Error (MSE): 0.0267
Root Mean Squared Error (RMSE): 0.1634
Summary saved: ../outputs/model_summaries/linear_model_summary_V1_correlated.txt


In [47]:
# Plot actual vs predicted
plt.Figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([0,1], [0,1], 'r--')
plt.title("Actual vs Predicted (test set)")
plt.xlabel("Actual Mobile Money Adoption Rate")
plt.ylabel("Predicted Mobile Money Adoption Rate")

# Saving the actual vs predicted plot
output_dir = "../outputs/charts/"
full_path_actual_vs_pred = os.path.join(output_dir, 'actual_vs_predicted.png')
plt.savefig(full_path_actual_vs_pred, bbox_inches='tight', dpi=300)
plt.close() # close the plot free up memory




In [ ]:
# Diagnostics - residuals & VIF

# Residuals
residuals = y_test - y_pred
sns.histplot(residuals, kde=True)
plt.title("Residuals Distribution")

# Saving the residuals distribution plot
output_dir = "../outputs/charts/"
full_path_residuals = os.path.join(output_dir, 'residuals_distribution.png')
plt.savefig(full_path_residuals, bbox_inches='tight', dpi=300)
plt.close() # close the plot free up memory

# Residuals vs predicted plot (scatter plot)
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuals vs Predicted")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.show()

# Variance Inflation Factor (VIF) to check multicollinearity

# --- 1. Prepare a temporary DataFrame for VIF calculation ---

# Create a copy of X_train features to avoid modifying the original
x_vif = X_train.copy()

# Ensure all data is numeric (float) for VIF calculation
x_vif = x_vif.astype(float)

# VIF requires an intercept (constant) term, so we add a constant column of 1s
x_vif['const'] = 1

# --- 2. Calculate VIF for each feature using a loop ---

# Create an empty DataFrame to store VIF results
vif_data = pd.DataFrame()

# Store the feature names in the DataFrame
vif_data['feature'] = x_vif.columns

# Calculate VIF for each feature
# We iterate over each column index (i) in x_vif from 0 to the number of columns
vif_data['VIF'] = [variance_inflation_factor(x_vif.values, i) for i in range(x_vif.shape[1])]

# --- 3. Format & display the VIF results ---

# Sort VIF values in descending order for better readability
vif_data = vif_data.sort_values(by='VIF', ascending=False)

# Display the VIF results
print("\n--- Variance Inflation Factor (VIF) ---")
print(vif_data)


##  Model Diagnosis: Multicollinearity (Dummy Variable Trap)
The Variance Inflation Factor (VIF) analysis revealed a severe issue: **Perfect Multicollinearity** (VIF=inf) among several categorical features, including `regionwb24_hi_High income` and the three`incomegroupwb24` categories.

This indicates we fell into the Dummy Variable Trap, where all categories of a single variable were retained after One-Hot Encoding. This creates a linear dependency with the intercept term, making the model coefficients unstable and mathematically unreliable.

VIF=inf is caused by **Dummy Variable Trap:** Sum of all binary categories Intercept.

## Corrective Action
To resolve this, we must remove one redundant category from each set of encoded features. This is equivalent to setting a "reference category" for the model.

**Step:** Drop one column from the incomegroupwb24 set and the redundant regionwb24_hi_High income column from both X_train and X_test before re-running the model diagnostics and training.

In [ ]:
# Create new training/test sets without the highly correlated columns
columns_to_drop = ['incomegroupwb24_Low income', 'regionwb24_hi_High income']
X_train_corrected = X_train.drop(columns=columns_to_drop, errors='ignore')
X_test_corrected = X_test.drop(columns=columns_to_drop, errors='ignore')
print("X_train_corrected and X_test_corrected are now defined.")

# Re-run the VIF check using X_train_corrected 
x_vif_corrected = X_train_corrected.copy()
x_vif_corrected = x_vif_corrected.astype(float)

x_vif_corrected['const'] = 1

data_vif = pd.DataFrame()
data_vif['feature'] = x_vif_corrected.columns
data_vif['VIF'] = [variance_inflation_factor(x_vif_corrected.values, i) for i in range(x_vif_corrected.shape[1])]
data_vif = data_vif.sort_values(by='VIF', ascending=False)
print("\n--- VIF After Removing Highly Correlated Features ---")
print(data_vif)

# Retrain the model with corrected features
model_corrected = LinearRegression()
model_corrected.fit(X_train_corrected, y_train)
y_pred_corrected = model_corrected.predict(X_test_corrected)

# Residuals
residuals_corrected = y_test - y_pred_corrected
sns.histplot(residuals_corrected, kde=True)
plt.title("Residuals Distribution (Corrected Model)")

# Saving the residuals distribution plot for corrected model
output_dir = "../outputs/charts/"
full_path_residuals_corrected = os.path.join(output_dir, 'lr_v2_residuals_distribution_corrected.png')
plt.savefig(full_path_residuals_corrected, bbox_inches='tight', dpi=300)
plt.close() # close the plot free up memory 

# Residuals vs predicted plot (scatter plot)
plt.scatter(y_pred_corrected, residuals_corrected, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuals vs Predicted (Corrected Model)")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")

# saving the residuals vs predicted plot for corrected model
output_dir = "../outputs/charts/"
full_path_residuals_vs_pred_corrected = os.path.join(output_dir, 'lr_v2_residuals_vs_predicted_corrected.png')
plt.savefig(full_path_residuals_vs_pred_corrected, bbox_inches='tight', dpi=300)
plt.close() # close the plot free up memory 

# Evaluate the corrected model performance
r2_corrected = r2_score(y_test, y_pred_corrected)
mse_corrected = mean_squared_error(y_test, y_pred_corrected)
rmse_corrected = np.sqrt(mse_corrected) 

# Print evaluation metrics for corrected model
print("\n--- Corrected Model Evaluation ---")
print(f"R-squared: {r2_corrected:.4f}")
print(f"Mean Squared Error (MSE): {mse_corrected:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_corrected:.4f}")

# Save corrected model evaluation summary to a text file
# Create the content for the text file
summary_text_corrected = (
    " --- Linear Regression V2: Clean Baseline ---\n"
    f"R-squared (Final Baseline): {r2_corrected:.4f}\n"
    f"Mean Squared Error (MSE): {mse_corrected:.4f}\n"
    f"Root Mean Squared Error (RMSE): {rmse_corrected:.4f}\n"
    "\nNOTE: This model has removed highly correlated features and serves as the best linear baseline."
)

# Define the path to save the summary
output_dir_txt = "../outputs/model_summaries/"
file_name = 'linear_model_summary_V2_baseline.txt'
full_path = os.path.join(output_dir_txt, file_name)
with open(full_path, 'w') as f:
    f.write(summary_text_corrected)



# Linear Regression Model Analysis: Diagnosis & Correction
The initial Linear Regression model was subjected to rigorous diagnostics to assess its stability and fit.

## Phase 1: Initial Diagnosis (Identifying Flaws)
| Diagnostic Tool | Observation | Conclusion |
| :--- | :---: | :--- |
| **VIF** | Several features displayed VIF=inf | **Severe multicollinearity (Dummy Variable Trap).** Coefficients were unstable and unreliable. |
| **Residuals plot** | Non-random "bow" pattern; Heteroscedasticity. | **Core Assumption of Linearity Violated.** Model is structurally inadequate. |
| **Actual vs. Predicted** | Predictions clustered horizontally (collapsed to the mean). | **Poor Predictive Power** |

## Phase 2: Corrective Action & Re-Evaluation

| Corrected Metric/Plot | Result | Conclusion|
| :--- | :---: | :--- |
| **Corrected VIF** | All scores $\text{VIF} < 3$ | **Model is stable.** The new coefficients are now reliable for statistical interpretation. |
|**Corrected R-squared** | $0.1182$ | **Predictive power remains poor.** Fixing the VIF did not improve the $R^2$. |
| **Corrected Residuals Plot** | Non-linear patterns and Heteroscedasticity persist. | **Linearity Assumption remains Violated.** A straight-line model is unsuitable for this data. |

## Conclusion and Next Step

The corrected Linear Regression model is **structurally sound** but **predictively insufficient** and **statistically invalid** due to the fundamentally non-linear relationship in the data.

We proceed to the final statistical step—extracting $P$-values from the OLS summary—to identify the statistically significant drivers before transitioning to a more powerful **Non-Linear Model** for improved predictive accuracy.


In [ ]:
# Run OLS regression for detailed summary
X_train_corrected_float = X_train_corrected.astype(float)
X_train_sm = sm.add_constant(X_train_corrected_float)  # Add constant term for intercept
ols_model = sm.OLS(y_train, X_train_sm).fit()

print("\n--- OLS Statistical Summary (Final Corrected Model) ---")
print(ols_model.summary())

# Save OLS summary to a text file
ols_summary_text = ols_model.summary().as_text()

output_dir = '../outputs/model_summaries/'
os.makedirs(output_dir, exist_ok=True) 

file_name = 'ols_pvalue_summary_inference.txt'
full_path = os.path.join(output_dir, file_name)

with open(full_path, 'w') as f:
    f.write(ols_summary_text)

print(f"\nOLS P-value summary saved to: {full_path}")

# Statistical Inference: Final OLS Regression Results

The OLS model was re-run using the corrected, stable feature set ($\text{X\_train\_corrected}$) to draw reliable statistical inferences about the drivers of Mobile Money Adoption.

## Key Performance Metrics

| Metric |	Value |	Interpretation |
| :--- | :---: | :--- |
| **R-squared ($\text{R}^2$)** | $0.127$ | The model explains $12.7\%$ of the variability in Mobile Money Adoption. This confirms the **poor predictive fit.** |
| **Prob (F-statistic)** | $7.38\text{e-}177$ (or $0.000$) | The model, as a whole, is **statistically significant** and better than a model with no predictors. |

## Feature Coefficients and Significance ($\alpha=0.05$)

The table below details the coefficient, $P$-value, and significance of each predictor, establishing their relationship with the Mobile Money Adoption Rate ($\text{mobileaccount\_t\_d}$).

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:left">Feature</th>
      <th style="text-align:center">Coefficient (coef)</th>
      <th style="text-align:center">P>|t|</th>
      <th style="text-align:center">Significance</th>
      <th style="text-align:left">Impact on Adoption</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><b>year</b></td>
      <td style="text-align:center">+0.0414</td>
      <td style="text-align:center">0.000</td>
      <td style="text-align:center"><b>Highly Significant</b></td>
      <td><b>Strong Positive.</b> Adoption increases by ≈ 4.14 percentage points per year.</td>
    </tr>
    <tr>
      <td><b>account_t_d</b></td>
      <td style="text-align:center">+0.0396</td>
      <td style="text-align:center">0.000</td>
      <td style="text-align:center"><b>Highly Significant</b></td>
      <td><b>Positive.</b> Higher traditional banking access is correlated with higher Mobile Money Adoption.</td>
    </tr>
    <tr>
      <td><b>internet</b></td>
      <td style="text-align:center">+0.0146</td>
      <td style="text-align:center">0.000</td>
      <td style="text-align:center"><b>Highly Significant</b></td>
      <td><b>Positive.</b> Higher internet penetration drives adoption.</td>
    </tr>
    <tr>
      <td><b>regionwb24_hi_Sub-Saharan Africa...</b></td>
      <td style="text-align:center">+0.0970</td>
      <td style="text-align:center">0.000</td>
      <td style="text-align:center"><b>Highly Significant</b></td>
      <td><b>Strongest Positive.</b> Adoption is highest in this region compared to the baseline (High Income regions).</td>
    </tr>
    <tr>
      <td><b>incomegroupwb24_Lower middle income</b></td>
      <td style="text-align:center">+0.0333</td>
      <td style="text-align:center">0.000</td>
      <td style="text-align:center"><b>Highly Significant</b></td>
      <td><b>Positive.</b> Higher adoption than the low-income baseline group.</td>
    </tr>
    <tr>
      <td><b>regionwb24_hi_Europe...</b></td>
      <td style="text-align:center">+0.0356</td>
      <td style="text-align:center">0.000</td>
      <td style="text-align:center"><b>Highly Significant</b></td>
      <td><b>Positive.</b> High adoption relative to the baseline.</td>
    </tr>
    <tr>
      <td><b>pop_adult</b></td>
      <td style="text-align:center">-0.0027</td>
      <td style="text-align:center">0.234</td>
      <td style="text-align:center">Not Significant</td>
      <td>Adult population size does not reliably influence the rate.</td>
    </tr>
    <tr>
      <td><b>regionwb24_hi_South Asia...</b></td>
      <td style="text-align:center">+0.0148</td>
      <td style="text-align:center">0.204</td>
      <td style="text-align:center">Not Significant</td>
      <td>Not a significant predictor in this model.</td>
    </tr>
  </tbody>
</table>




## Conclusion on Statistical Drivers

Despite the model's poor predictive power, the OLS analysis provides reliable insight into the statistical drivers:

1. **Strongest Drivers: Sub-Saharan Africa region** ($\text{coef}=+0.0970$), the **passage of time** ($\text{year}$), and **traditional banking access** ($\text{account\_t\_d}$) are the most robust positive predictors of Mobile Money Adoption.
2. Implications: The results suggest that Mobile Money often complements traditional banking and digital connectivity, rather than simply substituting them.

## Next Step

Given the poor $\text{R}^2$ ($0.127$) and the confirmed violation of model assumptions, the Linear Regression analysis is complete. The project must now transition to a **non-linear regression model** (such as Random Forest) to achieve a high-performing predictive model.


In [36]:
# -- RANDOM FOREST MODELING TO FOLLOW ---

import numpy as np
from sklearn.ensemble import RandomForestRegressor

In [ ]:
# --- 1. INITIALIZE THE MODEL ---
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# Train the Random Forest model
rf_model.fit(X_train_corrected, y_train)
print("Random Forest model trained.")

# Predict on the test set
y_rf_pred = rf_model.predict(X_test_corrected)

# Evaluate the Random Forest model performance
r2_rf = r2_score(y_test, y_rf_pred)
mse_rf = mean_squared_error(y_test, y_rf_pred)
rmse_rf = np.sqrt(mse_rf)

# Print evaluation metrics for Random Forest model
print("\n--- Random Forest Model Evaluation ---")   
print(f"RANDOM FOREST R-squared: {r2_rf:.4f}")
print(f"RANDOM FOREST RMSE: {rmse_rf:.4f}")

# Save Random Forest model evaluation summary to a text file
# Create the content for the text file
rf_summary_text = (
    "--- Random Forest Model Evaluation ---\n"
    f"R-squared: {r2_rf:.4f}\n"
    f"Root Mean Squared Error (RMSE): {rmse_rf:.4f}\n"
)

# Define the path to save the summary
output_dir = "../outputs/model_summaries/"
os.makedirs(output_dir, exist_ok=True)
file_name = 'random_forest_model_summary.txt'
full_path = os.path.join(output_dir, file_name)
with open(full_path, 'w') as f:
    f.write(rf_summary_text)    

**Conclusion:** The **Random Forest Regressor** is the correct model for this problem. It explains $68.04\%$ of the variability in Mobile Money Adoption, a massive improvement over the $12.7\%$ achieved by the linear model. The average prediction error has been cut almost in half.

**Next Step:** Understand which features the Random Forest model prioritized to achieve this high $\text{R}^2$. This will validate whether our statistically significant drivers from the OLS model are also the most important predictive features.

In [38]:
# --- EXTRACT FEATURE IMPORTANCES ---
# The feature importances are stored in the model object
importances = rf_model.feature_importances_

# Get feature names
feature_names = X_train_corrected.columns

# Create a DataFrame for feature importances
feature_importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# Sort features by importance
feature_importances = feature_importances.sort_values(by='Importance', ascending=False)

# Export the feature importance table to the output folder
feature_importances.to_csv('../outputs/rf_feature_importances_table.csv', index=False)

print("\n--- Random Forest Feature Importances ---")
print(feature_importances)


--- Random Forest Feature Importances ---
                                              Feature  Importance
1                                           pop_adult    0.311241
2                                         account_t_d    0.237174
3                                            internet    0.180382
0                                                year    0.118965
8   regionwb24_hi_Sub-Saharan Africa (excluding hi...    0.061631
10                incomegroupwb24_Upper middle income    0.031492
9                 incomegroupwb24_Lower middle income    0.017534
4   regionwb24_hi_Europe & Central Asia (excluding...    0.014157
5   regionwb24_hi_Latin America & Caribbean (exclu...    0.013741
6   regionwb24_hi_Middle East & North Africa (excl...    0.010883
7    regionwb24_hi_South Asia (excluding high income)    0.002800


In [ ]:
# Feature importance plot
data = {
    'Feature': ['pop_adult', 'account_t_d', 'internet', 'year', 'regionwb24_hi_Sub-Saharan Africa', 'incomegroupwb24_Upper middle income'],
    'Importance': [0.311241, 0.237174, 0.180382, 0.118965, 0.061631, 0.031492]
}
df_importance = pd.DataFrame(data)
df_importance = df_importance.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10,6))
colors = ['#1f77b4' if f != 'pop_adult' else '#d62728' for f in df_importance['Feature']]

sns.barplot(x='Importance', y='Feature', data=df_importance, palette=colors, orient='h')
plt.title('Random Forest: Final Predictive Drivers (R²=0.6804)', fontsize=16, pad=20)
plt.xlabel('Feature Importance Score', fontsize=12)
plt.ylabel('', fontsize=12)
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:.0%}'.format(x)))
plt.tight_layout()

# Saving the feature importance plot
output_dir = "../outputs/charts/"
full_path_feature_importance = os.path.join(output_dir, 'rf_feature_importance.png')
plt.savefig(full_path_feature_importance, bbox_inches='tight', dpi=300)
plt.close() # close the plot free up memory

In [55]:
# Create a new DataFrame specifically for Tableau with accurate Country column
# 1. Isolate the key identifiers (Country, Year) and the features (pop_adult, etc.)
#    We use df.index to make sure we align with the prediction array correctly.
df_identifiers = df[['countrynewwb', 'year']].copy()

# 2. Add the Predicted Adoption scores (y_full_pred) to the identifiers.
#    We assume the prediction array (y_full_pred) is already aligned with the original data index.
df_identifiers['Predicted_Adoption'] = y_full_pred

# 3. Add the other key features needed for Tableau from the original data
#    NOTE: We are getting the actual, un-scaled values here, which Tableau needs.
df_identifiers['account_t_d'] = df['account_t_d']
df_identifiers['mobileaccount_t_d'] = df['mobileaccount_t_d'] # This is Current Adoption
df_identifiers['internet'] = df['internet']
df_identifiers['pop_adult'] = df['pop_adult']


# 4. Filter the dataframe to only include the columns needed for Tableau
df_final_tableau_data = df_identifiers[['countrynewwb', 'year', 'account_t_d', 'pop_adult', 'mobileaccount_t_d', 'internet', 'Predicted_Adoption']].copy()

# 5. Export the new, accurate file
df_final_tableau_data.to_csv('../data/processed/tableau_ready_data.csv', index=False)

print("Tableau data saved to '../data/processed/tableau_ready_data.csv' with the Country column.")

Tableau data saved to '../data/processed/tableau_ready_data.csv' with the Country column.
